#We take economical indicators to do the hyperbolical embeding using matlab

In [9]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
import seaborn as sns
import geopandas as gpd
import matlab.engine
import warnings
warnings.filterwarnings('ignore')

# Try to import adjustText for non-overlapping labels
try:
    from adjustText import adjust_text
    ADJUSTTEXT_AVAILABLE = True
    print("adjustText library available - will use automatic label positioning")
except ImportError:
    ADJUSTTEXT_AVAILABLE = False
    print("adjustText not available - will use manual label positioning")
    print("To install: pip install adjustText")

# Setup paths
notebook_dir = Path.cwd()
matrices_dir = Path("matrices")  # Relative to Clean Code folder

print("Setup complete!")
print(f"Matrices directory: {matrices_dir}")
print(f"Exists: {matrices_dir.exists()}")

adjustText library available - will use automatic label positioning
Setup complete!
Matrices directory: matrices
Exists: True


## Configuration

In [10]:
# MATLAB paths
MATLAB_CE_PATH = r"D:\Tsinghua Classes and Papers\Network Science\migra-net-china\matlab\plot_hyperbolic_network_v2\matlab_scripts"

# Matrix types and their characteristics
MATRIX_TYPES = {
    'flows': {
        'description': 'Migration flow counts',
        'strength_method': 'sum',  # Sum of edge weights (node strength)
    },
    # 'edu_level': {
    #     'description': 'Education level of migrants',
    #     'strength_method': 'mean',  # Mean of edge characteristics
    # },
    # 'family_size': {
    #     'description': 'Family size of migrants',
    #     'strength_method': 'mean',  # Mean of edge characteristics
    # },
    # 'gender': {
    #     'description': 'Gender distribution of migrants',
    #     'strength_method': 'mean',  # Mean of edge characteristics
    # }
}

# Hyperbolic embedding parameters
HYPERBOLIC_PARAMS = {
    'rad_type': 'degree',
    'pre_weighting': 'EBC',  # Edge-Betweenness-Centrality
    'dim_red': 'ncISO',      # Non-centered Isomap
    'angular_adjustment': 'EA',  # Equidistant adjustment
    'dims': 2
}

# Years to process - ALL YEARS (1980-2016) + FULL (complete aggregated graph)
YEARS_TO_PROCESS = list(range(1980, 2017)) + ['full']

# Granularity levels - ALL THREE LEVELS
GRANULARITIES = ['economical_county', 'economical_prefecture', 'economical_province']

# Output directory
OUTPUT_DIR = Path('hyperbolic_outputs_by_characteristic')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  Matrix types: {list(MATRIX_TYPES.keys())}")
print(f"  Years: {YEARS_TO_PROCESS[:5]} ... {YEARS_TO_PROCESS[-5:]}")
print(f"  Total years: {len(YEARS_TO_PROCESS)} (37 years + 1 complete)")
print(f"  Granularities: {GRANULARITIES}")
print(f"  Total combinations: {len(GRANULARITIES)} × {len(YEARS_TO_PROCESS)} × {len(MATRIX_TYPES)} = {len(GRANULARITIES) * len(YEARS_TO_PROCESS) * len(MATRIX_TYPES)}")
print(f"  Note: Province 'full' will be skipped (power-law fitting issue)")
print(f"  Actual runs: {len(GRANULARITIES) * len(YEARS_TO_PROCESS) * len(MATRIX_TYPES) - 1}")
print(f"  Total visualizations: {(len(GRANULARITIES) * len(YEARS_TO_PROCESS) * len(MATRIX_TYPES) - 1) * 3}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Matrix types: ['flows']
  Years: [1980, 1981, 1982, 1983, 1984] ... [2013, 2014, 2015, 2016, 'full']
  Total years: 38 (37 years + 1 complete)
  Granularities: ['economical_county', 'economical_prefecture', 'economical_province']
  Total combinations: 3 × 38 × 1 = 114
  Note: Province 'full' will be skipped (power-law fitting issue)
  Actual runs: 113
  Total visualizations: 339
  Output: hyperbolic_outputs_by_characteristic


## Helper Functions

In [11]:
def compute_flow_strength_sum(adj_weighted_directed):
    """
    Compute flow strength as sum of edge weights (node strength).
    flow_strength = total flow through the node (in + out)
    """
    out_strength = adj_weighted_directed.sum(axis=1)
    in_strength = adj_weighted_directed.sum(axis=0)
    flow_strength = out_strength + in_strength
    return flow_strength


def compute_flow_strength_mean(adj_weighted_directed):
    """
    Compute flow strength as mean of edge characteristics.
    """
    out_degree = (adj_weighted_directed > 0).sum(axis=1)
    in_degree = (adj_weighted_directed > 0).sum(axis=0)
    total_degree = out_degree + in_degree
    
    out_char_sum = adj_weighted_directed.sum(axis=1)
    in_char_sum = adj_weighted_directed.sum(axis=0)
    total_char_sum = out_char_sum + in_char_sum
    
    flow_strength = np.zeros(len(total_degree))
    mask = total_degree > 0
    flow_strength[mask] = total_char_sum[mask] / total_degree[mask]
    
    return flow_strength


def compute_flow_strength(adj_weighted_directed, method='sum'):
    """Compute flow strength based on method (sum or mean)"""
    if method == 'sum':
        return compute_flow_strength_sum(adj_weighted_directed)
    elif method == 'mean':
        return compute_flow_strength_mean(adj_weighted_directed)
    else:
        raise ValueError(f"Unknown flow strength method: {method}")


def compute_polar_coordinates(adj_matrix, matlab_script_path, rad_type, pre_weighting, 
                                dim_red, angular_adjustment, dims=2):
    """
    Compute polar coordinates using MATLAB coalescent embedding.
    """
    np.fill_diagonal(adj_matrix, 0)
    
    # Ensure symmetric
    if not np.allclose(adj_matrix, adj_matrix.T):
        print("  Making matrix symmetric...")
        adj_matrix = np.maximum(adj_matrix, adj_matrix.T)
    
    print("  Starting MATLAB engine...")
    eng = matlab.engine.start_matlab()
    
    coords = None
    gamma = None
    
    try:
        eng.addpath(matlab_script_path, nargout=0)
        matlab_bgl_path = matlab_script_path + r'\matlab_bgl'
        eng.addpath(matlab_bgl_path, nargout=0)
        eng.addpath(eng.genpath(matlab_bgl_path), nargout=0)
        
        matlab_adj_matrix = matlab.double(adj_matrix.tolist())
        
        print("  Computing hyperbolic embedding...")
        matlab_coords, gamma = eng.coalescent_embedding_v2_1(
            matlab_adj_matrix,
            pre_weighting,
            dim_red,
            rad_type,
            angular_adjustment,
            dims,
            nargout=2
        )
        
        coords = np.array(matlab_coords)
        print(f"  Embedding complete! Gamma={gamma:.3f}")
        
    except Exception as e:
        print(f"  Error during MATLAB execution: {e}")
        raise
    finally:
        eng.quit()
    
    return coords, gamma


def detect_communities_louvain(G):
    """Detect communities using Louvain algorithm."""
    G_und = G.to_undirected()
    
    try:
        import community as community_louvain
        partition = community_louvain.best_partition(G_und, weight='weight')
    except ImportError:
        communities = nx.community.greedy_modularity_communities(G_und, weight='weight')
        partition = {}
        for i, community in enumerate(communities):
            for node in community:
                partition[node] = i
    
    return partition


def identify_core_nodes(G, percentile=90):
    """Identify core nodes based on degree (top percentile)."""
    degrees = dict(G.degree())
    threshold = np.percentile(list(degrees.values()), percentile)
    core_nodes = [node for node, deg in degrees.items() if deg >= threshold]
    return core_nodes

print("Helper functions defined!")

Helper functions defined!


## Visualization Functions

## Load Geographic Coordinates for Region Codes

In [12]:
# Load geo data to map region codes to Chinese names AND coordinates
geo_file_path = Path('df_sin_geo.csv')

print(f"Loading geo data from: {geo_file_path}")
df_geo = pd.read_csv(geo_file_path, on_bad_lines='skip')

# Create name mappings for all three granularities
code_to_name_prefecture = dict(zip(df_geo['Code_Perfecture'].astype(str), df_geo['Name_Perfecture']))
code_to_name_county = dict(zip(df_geo['Code_County'].astype(str), df_geo['Name_County']))
code_to_name_province = dict(zip(df_geo['Code_Province'].astype(str), df_geo['Name_Province']))

# Load coordinate files for each granularity
print(f"\nLoading coordinate data...")

# County coordinates (from migration data)
data_csv_path = r"D:\Tsinghua Classes and Papers\Network Science\migra-net-china\src\data\data.csv"
df_migration = pd.read_csv(data_csv_path)
region_coord_county = df_migration[['hometown_code', 'hometown_lon', 'hometown_lat']].dropna()
region_coord_county = region_coord_county.drop_duplicates(subset=['hometown_code'])
region_coord_county = region_coord_county.rename(columns={
    'hometown_code': 'code',
    'hometown_lon': 'lon',
    'hometown_lat': 'lat'
})
region_coord_county['code'] = region_coord_county['code'].astype(str)
print(f"  County coordinates: {len(region_coord_county)} regions")

# Prefecture coordinates
prefecture_file = Path('df_geo_prefectures.csv')
region_coord_prefecture = pd.read_csv(prefecture_file)
region_coord_prefecture['code'] = region_coord_prefecture['code'].astype(str)
print(f"  Prefecture coordinates: {len(region_coord_prefecture)} regions")

# Province coordinates
province_file = Path('df_geo_provinces.csv')
region_coord_province = pd.read_csv(province_file)
region_coord_province['code'] = region_coord_province['code'].astype(str)
print(f"  Province coordinates: {len(region_coord_province)} regions")

# Create a dictionary to map granularity to coordinate dataframe
coord_mapping = {
    'economical_county': region_coord_county,
    'economical_prefecture': region_coord_prefecture,
    'economical_province': region_coord_province
}

# Create a dictionary to map granularity to name function
name_mapping = {
    'economical_county': lambda code: code_to_name_county.get(str(code), str(code)),
    'economical_prefecture': lambda code: code_to_name_prefecture.get(str(code), str(code)),
    'economical_province': lambda code: code_to_name_province.get(str(code), str(code))
}

# Helper functions to get names from codes (backward compatibility)
def get_city_name(code):
    """Convert prefecture code to readable Chinese name"""
    code_str = str(code)
    return code_to_name_prefecture.get(code_str, code_str)

def get_county_name(code):
    """Convert county code to readable Chinese name"""
    code_str = str(code)
    return code_to_name_county.get(code_str, code_str)

def get_province_name(code):
    """Convert province code to readable Chinese name"""
    code_str = str(code)
    return code_to_name_province.get(code_str, code_str)

# Generic function to get name based on granularity
def get_region_name(code, granularity):
    """Get region name based on granularity"""
    return name_mapping[granularity](code)

# Configure matplotlib to display Chinese characters
from matplotlib.font_manager import FontManager

fm = FontManager()
chinese_fonts = []

# Search for common Chinese fonts
font_names_to_try = ['Microsoft YaHei', 'SimHei', 'STSong', 'KaiTi', 'FangSong', 
                      'PingFang SC', 'Heiti SC', 'WenQuanYi Zen Hei', 'Noto Sans CJK SC']

for font_name in font_names_to_try:
    if any(font_name.lower() in f.name.lower() for f in fm.ttflist):
        chinese_fonts.append(font_name)

if chinese_fonts:
    plt.rcParams['font.sans-serif'] = chinese_fonts + ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    print(f"\nFound Chinese fonts: {chinese_fonts[:3]}")
    print(f"Using font: {chinese_fonts[0]}")
else:
    print("\nWarning: No Chinese fonts found. Will use codes in plots.")

print(f"\nLoaded geo data:")
print(f"  {len(code_to_name_prefecture)} prefecture codes mapped to names")
print(f"  {len(code_to_name_county)} county codes mapped to names")
print(f"  {len(code_to_name_province)} province codes mapped to names")
print(f"\nSample prefecture mappings:")
for i, (code, name) in enumerate(list(code_to_name_prefecture.items())[:5]):
    print(f"  {code} -> {name}")

Loading geo data from: df_sin_geo.csv

Loading coordinate data...
  County coordinates: 2665 regions
  Prefecture coordinates: 331 regions
  Province coordinates: 31 regions

Found Chinese fonts: ['Microsoft YaHei', 'SimHei', 'KaiTi']
Using font: Microsoft YaHei

Loaded geo data:
  347 prefecture codes mapped to names
  2901 county codes mapped to names
  34 province codes mapped to names

Sample prefecture mappings:
  1101 -> 北京市
  1201 -> 天津市
  1307 -> 张家口市
  1308 -> 承德市
  1301 -> 石家庄市


In [13]:
def plot_hyperbolic_network(adj_matrix, coords, flow_strength, year, matrix_type, region_codes, granularity, output_path=None):
    """
    Plot network in hyperbolic space colored by flow strength with top 10 nodes labeled.
    """
    # Convert polar to cartesian
    coords_x = coords[:, 1] * np.cos(coords[:, 0])
    coords_y = coords[:, 1] * np.sin(coords[:, 0])

    # Create figure with extra space on left for legend
    fig, ax = plt.subplots(figsize=(20, 14))
    ax.set_facecolor('white')

    # Plot edges
    G = nx.from_numpy_array(adj_matrix)
    pos = {i: (coords_x[i], coords_y[i]) for i in range(len(coords_x))}
    nx.draw_networkx_edges(G, pos, alpha=0.05, width=0.1, edge_color='gray', ax=ax)

    # Color nodes by flow strength (log scale)
    strength_log = np.log10(flow_strength + 1)

    # Create colormap
    from matplotlib.colors import LinearSegmentedColormap
    colors_list = ['blue', 'cyan', 'yellow', 'red']
    cmap = LinearSegmentedColormap.from_list('strength_cmap', colors_list, N=100)

    # Plot nodes
    scatter = ax.scatter(coords_x, coords_y, s=50, c=strength_log, cmap=cmap,
                        edgecolors='k', linewidths=0.5, alpha=0.8, zorder=100)

    # Label top 10 nodes by flow strength with names appropriate to granularity
    top_10_indices = np.argsort(flow_strength)[-10:][::-1]
    legend_entries = []
    
    for rank, idx in enumerate(top_10_indices, 1):
        region_code = region_codes[idx]
        region_name = get_region_name(region_code, granularity)
        
        # Add small label on plot
        ax.annotate(
            f'{region_name}\n({flow_strength[idx]:.0f})',
            xy=(coords_x[idx], coords_y[idx]),
            xytext=(10, 10),
            textcoords='offset points',
            fontsize=7,
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7, edgecolor='black', linewidth=1),
            zorder=200,
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.1', lw=1, color='black')
        )
        
        # Add to legend list
        legend_entries.append(f'{rank}. {region_name} ({flow_strength[idx]:.0f})')

    # Formatting
    radius = coords[:, 1].max()
    ax.set_xlim(-radius*1.1, radius*1.1)
    ax.set_ylim(-radius*1.1, radius*1.1)
    ax.set_aspect('equal')
    ax.axis('off')

    # Title
    matrix_desc = MATRIX_TYPES[matrix_type]['description']
    ax.set_title(f'中国人口迁移网络 {year} - {granularity}\n{matrix_desc}',
                 fontsize=18, fontweight='bold', pad=20)

    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('log10(Flow Strength + 1)', fontsize=12)

    # Adjust layout to make room for legend on the left
    fig.subplots_adjust(left=0.25, right=0.95, top=0.95, bottom=0.05)

    # Add legend box OUTSIDE on the left
    legend_text = 'Top 10 Regions:\n' + '\n'.join(legend_entries)
    ax.text(-0.15, 0.5, legend_text, transform=ax.transAxes,
           fontsize=10, verticalalignment='center',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, pad=1))

    if output_path:
        plt.savefig(output_path, dpi=200)
        print(f"  Saved: {output_path}")

    plt.close()

    return fig


def plot_teu_on_china_map(flow_strength, region_coords, year, matrix_type, granularity, output_path=None):
    """Plot flow strength values on geographic map of China with top 10 nodes labeled."""
    import geopandas as gpd
    from matplotlib.colors import LinearSegmentedColormap

    # Load China map
    china_map_path = Path("../src/data/china_provinces.json")
    if not china_map_path.exists():
        print(f"    ⚠ China map not found at {china_map_path}")
        return None

    china_map = gpd.read_file(china_map_path)

    # Create colormap
    colors_list = ['blue', 'cyan', 'yellow', 'red']
    cmap = LinearSegmentedColormap.from_list('strength_cmap', colors_list, N=100)

    # Prepare data
    strength_log = np.log10(flow_strength + 1)
    top_10_indices = np.argsort(flow_strength)[-10:][::-1]

    # Create figure with extra width for legend
    fig, ax = plt.subplots(figsize=(24, 14))

    # Plot China map
    china_map.plot(ax=ax, color='#f0f0f0', edgecolor='#d9d9d9', linewidth=0.8)

    # Plot all nodes colored by flow strength
    scatter = ax.scatter(
        region_coords['lon'],
        region_coords['lat'],
        c=strength_log,
        s=flow_strength * 3 + 50,
        cmap=cmap,
        alpha=0.7,
        edgecolors='black',
        linewidths=1,
        zorder=100
    )

    # Add labels for top 10 nodes with names appropriate to granularity
    legend_entries = []
    
    for rank, idx in enumerate(top_10_indices, 1):
        if idx < len(region_coords):
            region_info = region_coords.iloc[idx]
            lon, lat = region_info['lon'], region_info['lat']
            region_code = region_info['code']
            region_name = get_region_name(region_code, granularity)

            if pd.notna(lon) and pd.notna(lat):
                # Add small label on map
                ax.annotate(
                    f'{rank}. {region_name}\n({flow_strength[idx]:.0f})',
                    xy=(lon, lat),
                    xytext=(12, 12),
                    textcoords='offset points',
                    fontsize=8,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.75, edgecolor='black', linewidth=1),
                    zorder=200,
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', lw=1.5, color='black')
                )
                
                # Add to legend list
                legend_entries.append(f'{rank}. {region_name} ({flow_strength[idx]:.0f})')

    # Formatting
    matrix_desc = MATRIX_TYPES[matrix_type]['description']
    ax.set_title(f'中国人口迁移网络 {year} - {granularity}\n{matrix_desc} - 地理分布',
                 fontsize=18, fontweight='bold', pad=20)
    ax.set_xlim(73, 136)
    ax.set_ylim(18, 54)
    ax.axis('off')

    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('log10(Flow Strength + 1)', fontsize=12)

    # Adjust layout to make room for legend on the left
    fig.subplots_adjust(left=0.15, right=0.95, top=0.95, bottom=0.05)

    # Add legend box OUTSIDE on the left
    legend_text = 'Top 10 Regions:\n' + '\n'.join(legend_entries)
    ax.text(-0.10, 0.5, legend_text, transform=ax.transAxes,
           fontsize=11, verticalalignment='center',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, pad=1))

    if output_path:
        plt.savefig(output_path, dpi=200)
        print(f"  Saved: {output_path}")

    plt.close()

    return fig

print("Visualization functions defined!")

Visualization functions defined!


In [14]:
def plot_hyperbolic_network_with_communities(adj_matrix, coords, flow_strength, year, matrix_type,
                                             region_codes, partition, core_nodes, granularity, output_path=None):
    """
    Plot network in hyperbolic space - styled like GLSN paper figure.
    - Nodes colored by community
    - Core nodes with thick black borders
    - Intra-core edges in dark gray, others in light gray
    - Clean labeling of top nodes
    """
    # Convert polar to cartesian
    coords_x = coords[:, 1] * np.cos(coords[:, 0])
    coords_y = coords[:, 1] * np.sin(coords[:, 0])

    # Create figure
    fig, ax = plt.subplots(figsize=(20, 18))
    ax.set_facecolor('white')

    # Handle partition format
    if isinstance(partition, dict):
        partition_list = [partition.get(i, -1) for i in range(len(coords_x))]
    else:
        partition_list = partition
    
    communities = sorted(set(partition_list))
    n_communities = len(communities)

    # Analyze communities
    community_info = {}
    for comm_id in communities:
        comm_nodes = [i for i, c in enumerate(partition_list) if c == comm_id]
        if len(comm_nodes) == 0:
            continue
        
        total_flow = sum(flow_strength[n] for n in comm_nodes)
        core_in_comm = [n for n in comm_nodes if n in core_nodes]
        
        community_info[comm_id] = {
            'size': len(comm_nodes),
            'total_flow': total_flow,
            'n_core': len(core_in_comm),
            'nodes': comm_nodes
        }
    
    # Assign colors to communities
    if n_communities <= 10:
        colors = plt.cm.tab10(np.linspace(0, 1, 10))
    elif n_communities <= 20:
        colors = plt.cm.tab20(np.linspace(0, 1, 20))
    else:
        colors = plt.cm.hsv(np.linspace(0, 0.95, n_communities))
    
    community_colors = {comm_id: colors[i % len(colors)] for i, comm_id in enumerate(communities)}

    # Build graph for edge drawing
    G = nx.from_numpy_array(adj_matrix)
    pos = {i: (coords_x[i], coords_y[i]) for i in range(len(coords_x))}
    
    # Draw edges with different styles
    # 1. Intra-core edges (dark gray, thicker)
    intra_core_edges = [(i, j) for i, j in G.edges() if i in core_nodes and j in core_nodes]
    nx.draw_networkx_edges(G, pos, edgelist=intra_core_edges, 
                          alpha=0.4, width=0.3, edge_color='#333333', ax=ax)
    
    # 2. Other edges (very light gray, thin)
    other_edges = [(i, j) for i, j in G.edges() if not (i in core_nodes and j in core_nodes)]
    nx.draw_networkx_edges(G, pos, edgelist=other_edges,
                          alpha=0.05, width=0.1, edge_color='#CCCCCC', ax=ax)

    # Plot nodes with community colors
    # NON-CORE nodes: regular circles with thin white border
    for comm_id in communities:
        comm_nodes = community_info[comm_id]['nodes']
        non_core_in_comm = [n for n in comm_nodes if n not in core_nodes]
        
        if non_core_in_comm:
            x_vals = [coords_x[i] for i in non_core_in_comm]
            y_vals = [coords_y[i] for i in non_core_in_comm]
            sizes = [50 + flow_strength[i]/max(flow_strength)*120 if max(flow_strength) > 0 else 50 
                    for i in non_core_in_comm]
            
            ax.scatter(x_vals, y_vals, s=sizes, 
                      c=[community_colors[comm_id]],
                      marker='o', 
                      edgecolors='white', 
                      linewidths=0.5,
                      alpha=0.85, 
                      zorder=100)
    
    # CORE nodes: circles with THICK BLACK BORDER (like the paper)
    core_x = [coords_x[i] for i in core_nodes]
    core_y = [coords_y[i] for i in core_nodes]
    core_sizes = [100 + flow_strength[i]/max(flow_strength)*200 if max(flow_strength) > 0 else 100 
                  for i in core_nodes]
    core_colors = [community_colors[partition_list[i]] for i in core_nodes]
    
    ax.scatter(core_x, core_y, s=core_sizes, 
              c=core_colors,
              marker='o', 
              edgecolors='black',  # Thick black border for core
              linewidths=3.0,  # Thick border
              alpha=0.95, 
              zorder=200)

    # Label only top nodes (similar to paper's figure b)
    # Find top N nodes overall by flow strength
    N_LABELS = min(15, len(region_codes))
    top_nodes_indices = np.argsort(flow_strength)[-N_LABELS:][::-1]
    
    for idx in top_nodes_indices:
        region_name = get_region_name(region_codes[idx], granularity)
        comm_id = partition_list[idx]
        
        # Use community color for label background
        ax.annotate(
            region_name,
            xy=(coords_x[idx], coords_y[idx]),
            xytext=(12, 12),
            textcoords='offset points',
            fontsize=9,
            fontweight='bold' if idx in core_nodes else 'normal',
            color='black',
            bbox=dict(boxstyle='round,pad=0.4', 
                     facecolor=community_colors[comm_id], 
                     alpha=0.8,
                     edgecolor='black' if idx in core_nodes else 'gray',
                     linewidth=1.5 if idx in core_nodes else 0.8),
            zorder=300,
            arrowprops=dict(arrowstyle='->', 
                          lw=1.5 if idx in core_nodes else 1.0,
                          color='black')
        )

    # Formatting
    radius = coords[:, 1].max()
    ax.set_xlim(-radius*1.15, radius*1.15)
    ax.set_ylim(-radius*1.15, radius*1.15)
    ax.set_aspect('equal')
    ax.axis('off')

    # Title
    matrix_desc = MATRIX_TYPES[matrix_type]['description']
    ax.set_title(f'{granularity} - {year}\n{matrix_desc} - Community Structure',
                 fontsize=16, fontweight='bold', pad=20)

    # Adjust layout BEFORE creating legend to avoid affine transformation errors
    fig.subplots_adjust(left=0.05, right=0.75, top=0.93, bottom=0.05)
    
    # Sort communities by total flow for legend
    sorted_comms = sorted(community_info.items(), 
                         key=lambda x: x[1]['total_flow'], 
                         reverse=True)
    
    legend_lines = [
        f"NETWORK STRUCTURE",
        f"="*50,
        f"",
        f"Total nodes: {len(region_codes)}",
        f"Core nodes: {len(core_nodes)} (thick black border)",
        f"Communities: {n_communities}",
        f"",
        f"Edges:",
        f"  - Dark gray: Intra-core connections",
        f"  - Light gray: Other connections",
        f"",
        f"="*50,
        f"TOP COMMUNITIES (by total flow):",
        f"="*50,
    ]
    
    # Show top 10 communities
    for rank, (comm_id, info) in enumerate(sorted_comms[:10], 1):
        legend_lines.append(f"")
        legend_lines.append(f"Module {comm_id} (Rank {rank})")
        legend_lines.append(f"  Nodes: {info['size']} | Core: {info['n_core']}")
        legend_lines.append(f"  Total flow: {info['total_flow']:.0f}")
        
        # Show top 2 nodes
        comm_flows = [(n, flow_strength[n]) for n in info['nodes']]
        comm_flows.sort(key=lambda x: x[1], reverse=True)
        
        for i, (node_idx, flow) in enumerate(comm_flows[:2], 1):
            name = get_region_name(region_codes[node_idx], granularity)
            core_mark = " [CORE]" if node_idx in core_nodes else ""
            legend_lines.append(f"  {i}. {name}{core_mark}")
    
    if n_communities > 10:
        legend_lines.append(f"")
        legend_lines.append(f"... + {n_communities - 10} more communities")
    
    legend_text = '\n'.join(legend_lines)
    
    # Explicitly use Chinese font for legend - use font properties
    from matplotlib import font_manager
    # Use the same font we configured for Chinese support
    font_prop = font_manager.FontProperties(family='sans-serif', size=9)
    
    ax.text(1.02, 0.5, legend_text, transform=ax.transAxes,
           fontproperties=font_prop,
           verticalalignment='center',
           bbox=dict(boxstyle='round,pad=1.0', 
                    facecolor='white', 
                    alpha=0.95, 
                    edgecolor='gray',
                    linewidth=1.0))

    if output_path:
        # Save without bbox_inches='tight' to avoid affine transformation error
        plt.savefig(output_path, dpi=300)
        print(f"  Saved: {output_path}")

    plt.close()

    return fig

print("Community visualization - with proper Chinese font support!")

Community visualization - with proper Chinese font support!


## Main Processing Loop

In [15]:
# Store all results - RESET to recalculate everything
all_results = []

# Process each granularity
for granularity in GRANULARITIES:
    print(f"\n{'#'*100}")
    print(f"# GRANULARITY: {granularity.upper()}")
    print(f"{'#'*100}")
    
    # Get the appropriate coordinate mapping for this granularity
    region_coord_mapping = coord_mapping[granularity]
    
    for year in YEARS_TO_PROCESS:
        # SKIP 'full' for province level due to power-law fitting issues
        if granularity == 'economical_province' and year == 'full':
            print(f"\n⚠️ Skipping {granularity} - {year} (power-law fitting not possible with few nodes)")
            continue
        
        print(f"\n{'='*80}")
        print(f"Processing {granularity} - Year {year}")
        print(f"{'='*80}")
        
        # Process each matrix type
        for matrix_type, config in MATRIX_TYPES.items():
            print(f"\n  {'-'*60}")
            print(f"  Matrix Type: {matrix_type.upper()}")
            print(f"  Description: {config['description']}")
            print(f"  Strength method: {config['strength_method']}")
            print(f"  {'-'*60}")
            
            # Load matrix for this type
            matrix_file = matrices_dir / matrix_type / f'{granularity}_{year}.csv'
            
            if not matrix_file.exists():
                print(f"    ⚠ File not found: {matrix_file}")
                continue
            
            print(f"    Loading matrix...")
            df_matrix = pd.read_csv(matrix_file, index_col=0)
            adj_weighted_directed = df_matrix.values
            region_codes = df_matrix.index.astype(str).tolist()
            
            print(f"    Matrix shape: {adj_weighted_directed.shape}")
            print(f"    Number of regions: {len(region_codes)}")
            
            # Create region coordinate DataFrame for this year and granularity
            region_coords_year = []
            for region_code in region_codes:
                coord_info = region_coord_mapping[region_coord_mapping['code'] == region_code]
                if len(coord_info) > 0:
                    region_coords_year.append({
                        'code': region_code,
                        'lon': coord_info.iloc[0]['lon'],
                        'lat': coord_info.iloc[0]['lat']
                    })
                else:
                    region_coords_year.append({
                        'code': region_code,
                        'lon': np.nan,
                        'lat': np.nan
                    })
            
            region_coords_df = pd.DataFrame(region_coords_year)
            print(f"    Mapped {len(region_coords_df.dropna())} regions to geographic coordinates")
            
            # Compute flow strength
            strength_method = config['strength_method']
            flow_strength = compute_flow_strength(adj_weighted_directed, method=strength_method)
            print(f"    Flow strength computed: min={flow_strength.min():.2f}, max={flow_strength.max():.2f}, mean={flow_strength.mean():.2f}")
            
            # Show top 10 regions by flow strength (using granularity-appropriate names)
            top_10_indices = np.argsort(flow_strength)[-10:][::-1]
            print(f"\n    Top 10 regions by flow strength:")
            for rank, idx in enumerate(top_10_indices, 1):
                region_code = region_codes[idx]
                region_name = get_region_name(region_code, granularity)
                print(f"      {rank}. {region_name} ({region_code}): {flow_strength[idx]:.2f}")
            
            # Convert to unweighted undirected for hyperbolic embedding
            adj_unweighted_directed = (adj_weighted_directed > 0).astype(int)
            adj_matrix = np.maximum(adj_unweighted_directed, adj_unweighted_directed.T)
            
            num_nodes = adj_matrix.shape[0]
            num_edges = np.count_nonzero(adj_matrix) // 2
            print(f"\n    Unweighted undirected: {num_nodes} nodes, {num_edges} edges")
            
            # Compute hyperbolic coordinates
            print(f"    Computing hyperbolic embedding...")
            try:
                coords, gamma = compute_polar_coordinates(
                    adj_matrix, 
                    MATLAB_CE_PATH,
                    **HYPERBOLIC_PARAMS
                )
                print(f"    Hyperbolic coords shape: {coords.shape}")
            except Exception as e:
                print(f"    ⚠️ Error computing hyperbolic embedding: {e}")
                print(f"    Skipping {granularity} - {year} - {matrix_type}")
                continue
            
            # Build NetworkX graph for community detection
            G = nx.from_numpy_array(adj_weighted_directed, create_using=nx.DiGraph)
            
            # Detect communities
            print("    Detecting communities...")
            partition = detect_communities_louvain(G)
            n_communities = len(set(partition.values()))
            print(f"    Found {n_communities} communities")
            
            # Identify core nodes
            core_nodes = identify_core_nodes(G, percentile=90)
            print(f"    Identified {len(core_nodes)} core nodes (top 10% by degree)")
            
            # Convert partition dict to list for indexing
            partition_list = [partition.get(i, -1) for i in range(num_nodes)]
            
            # Create visualizations
            print(f"\n    Creating visualizations...")
            
            # 1. Hyperbolic visualization with flow strength coloring (EXISTING)
            print(f"      - Hyperbolic network visualization (flow strength)...")
            output_file = OUTPUT_DIR / f'{granularity}_{year}_{matrix_type}_hyperbolic_flow.png'
            plot_hyperbolic_network(
                adj_matrix, coords, flow_strength, year, matrix_type, region_codes,
                granularity, output_path=output_file
            )
            
            # 2. Hyperbolic visualization with community coloring (NEW)
            print(f"      - Hyperbolic network visualization (communities)...")
            output_file_comm = OUTPUT_DIR / f'{granularity}_{year}_{matrix_type}_hyperbolic_communities.png'
            plot_hyperbolic_network_with_communities(
                adj_matrix, coords, flow_strength, year, matrix_type, region_codes,
                partition_list, core_nodes, granularity,
                output_path=output_file_comm
            )
            
            # 3. Geographic map with flow strength overlay (EXISTING)
            print(f"      - Geographic map visualization...")
            region_coords_valid = region_coords_df.dropna()
            if len(region_coords_valid) > 0:
                # Align flow strength values with valid coordinates
                flow_strength_for_map = np.array([
                    flow_strength[i] for i, rc in enumerate(region_codes) 
                    if rc in region_coords_valid['code'].values
                ])
                
                map_output_file = OUTPUT_DIR / f'{granularity}_{year}_{matrix_type}_map_flow_strength.png'
                plot_teu_on_china_map(
                    flow_strength_for_map, 
                    region_coords_valid,
                    year,
                    matrix_type,
                    granularity,
                    output_path=map_output_file
                )
            else:
                print(f"      ⚠ No valid coordinates for geographic map")
            
            # Store results
            for i in range(num_nodes):
                all_results.append({
                    'granularity': granularity,
                    'year': year,
                    'matrix_type': matrix_type,
                    'node_id': i,
                    'region_code': region_codes[i],
                    'region_name': get_region_name(region_codes[i], granularity),
                    'theta': coords[i, 0],
                    'r': coords[i, 1],
                    'x': coords[i, 1] * np.cos(coords[i, 0]),
                    'y': coords[i, 1] * np.sin(coords[i, 0]),
                    'flow_strength': flow_strength[i],
                    'strength_method': strength_method,
                    'community': partition.get(i, -1),
                    'is_core': i in core_nodes,
                    'gamma': gamma,
                    'lon': region_coords_df.iloc[i]['lon'] if i < len(region_coords_df) else np.nan,
                    'lat': region_coords_df.iloc[i]['lat'] if i < len(region_coords_df) else np.nan,
                    'n_communities': n_communities
                })
            
            print(f"\n    ✓ Matrix type {matrix_type} complete!")
        
        print(f"\n  ✓ Year {year} complete!")
    
    print(f"\n{'#'*100}")
    print(f"# ✓ Granularity {granularity.upper()} COMPLETE!")
    print(f"{'#'*100}")

print(f"\n{'='*100}")
print("ALL PROCESSING COMPLETE!")
print(f"Total results collected: {len(all_results)}")
print(f"{'='*100}")


####################################################################################################
# GRANULARITY: ECONOMICAL_COUNTY
####################################################################################################

Processing economical_county - Year 1980

  ------------------------------------------------------------
  Matrix Type: FLOWS
  Description: Migration flow counts
  Strength method: sum
  ------------------------------------------------------------
    Loading matrix...
    Matrix shape: (2697, 2697)
    Number of regions: 2697
    Mapped 2657 regions to geographic coordinates
    Flow strength computed: min=0.00, max=4.00, mean=0.10

    Top 10 regions by flow strength:
      1. 黄浦区 (310101): 4.00
      2. 临夏县 (622921): 3.00
      3. 柳北区 (450205): 3.00
      4. 蓬溪县 (510921): 3.00
      5. 长寿区 (500115): 2.00
      6. 浦东新区 (310115): 2.00
      7. 临夏市 (622901): 2.00
      8. 海拉尔区 (150702): 2.00
      9. 东城区 (110101): 2.00
      10. 张湾区 (420303): 2.00

   

## Export Results

In [16]:
# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Save comprehensive CSV with ALL data
output_csv = OUTPUT_DIR / 'hyperbolic_embeddings_all_granularities_years.csv'
results_df.to_csv(output_csv, index=False)

print(f"\n{'='*80}")
print("Results Summary")
print(f"{'='*80}")
print(f"Total records: {len(results_df):,}")
print(f"Granularities: {sorted(results_df['granularity'].unique())}")

# Handle mixed types in year column (integers and 'full')
years_list = results_df['year'].unique()
years_int = sorted([y for y in years_list if isinstance(y, int)])
years_str = sorted([y for y in years_list if isinstance(y, str)])
print(f"Years: {years_int + years_str}")

print(f"Matrix types: {sorted(results_df['matrix_type'].unique())}")
print(f"\nColumns:")
for col in results_df.columns:
    print(f"  - {col}")
print(f"\nSaved to: {output_csv}")
print(f"File size: {output_csv.stat().st_size / (1024*1024):.2f} MB")

# Display summary statistics by granularity
print(f"\n{'='*80}")
print("Summary by Granularity and Year")
print(f"{'='*80}")
summary_by_gran = results_df.groupby(['granularity', 'year']).agg({
    'node_id': 'count',
    'flow_strength': ['min', 'max', 'mean'],
    'n_communities': 'first',
    'is_core': 'sum'
}).round(2)
summary_by_gran.columns = ['num_nodes', 'flow_min', 'flow_max', 'flow_mean', 'n_communities', 'num_core_nodes']
print(summary_by_gran.head(20))

# Display sample
print(f"\nSample data (first 10 rows):")
results_df.head(10)


Results Summary
Total records: 116,211
Granularities: ['economical_county', 'economical_prefecture', 'economical_province']
Years: [1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 'full']
Matrix types: ['flows']

Columns:
  - granularity
  - year
  - matrix_type
  - node_id
  - region_code
  - region_name
  - theta
  - r
  - x
  - y
  - flow_strength
  - strength_method
  - community
  - is_core
  - gamma
  - lon
  - lat
  - n_communities

Saved to: hyperbolic_outputs_by_characteristic\hyperbolic_embeddings_all_granularities_years.csv
File size: 21.36 MB

Summary by Granularity and Year
                        num_nodes  flow_min  flow_max  flow_mean  \
granularity       year                                             
economical_county 1980       2697       0.0       4.0       0.10   
                  1981     

,granularity,year,matrix_type,node_id,region_code,region_name,theta,r,x,y,flow_strength,strength_method,community,is_core,gamma,lon,lat,n_communities
0,economical_county,1980,flows,0,360681,贵溪市,0.000000,14.145614,14.145614,0.000000,0.0,sum,0,True,3.92,117.186973,28.188428,2564
1,economical_county,1980,flows,1,360111,青山湖区,0.002330,14.148450,14.148411,0.032962,0.0,sum,1,True,3.92,115.905297,28.719082,2564
2,economical_county,1980,flows,2,360402,濂溪区,0.004659,14.151274,14.151121,0.065936,0.0,sum,2,True,3.92,116.039436,29.634605,2564
3,economical_county,1980,flows,3,440305,南山区,0.006989,14.154087,14.153741,0.098923,0.0,sum,3,True,3.92,113.937903,22.554902,2564
4,economical_county,1980,flows,4,511623,邻水县,0.009319,12.506056,12.505513,0.116539,1.0,sum,4,True,3.92,106.991830,30.258922,2564
5,economical_county,1980,flows,5,350503,丰泽区,0.011648,14.156889,14.155928,0.164902,0.0,sum,5,True,3.92,118.617882,24.922059,2564
6,economical_county,1980,flows,6,511702,通川区,0.013978,14.159678,14.158295,0.197920,0.0,sum,6,True,3.92,107.432388,31.362205,2564
7,economical_county,1980,flows,7,510107,武侯区,0.016308,12.536502,12.534835,0.204434,1.0,sum,7,True,3.92,104.022906,30.610118,2564
8,economical_county,1980,flows,8,230229,克山县,0.018638,14.162457,14.159997,0.263938,0.0,sum,8,True,3.92,125.670668,48.168083,2564
9,economical_county,1980,flows,9,210804,鲅鱼圈区,0.020967,14.165224,14.162111,0.296984,0.0,sum,9,True,3.92,122.171994,40.259183,2564


## Summary Statistics and Comparison

In [27]:
# Summary by granularity, year and matrix type
summary = results_df.groupby(['granularity', 'year', 'matrix_type']).agg({
    'node_id': 'count',
    'flow_strength': ['min', 'max', 'mean', 'std'],
    'n_communities': 'first',
    'is_core': 'sum',
    'gamma': 'first'
}).round(3)

summary.columns = ['num_nodes', 'strength_min', 'strength_max', 'strength_mean', 'strength_std', 
                   'num_communities', 'num_core_nodes', 'gamma']

print(f"\n{'='*80}")
print("Summary by Granularity, Year and Matrix Type:")
print(f"{'='*80}")
print(summary.head(50))

# Save summary
summary_csv = OUTPUT_DIR / 'hyperbolic_summary_by_granularity_year.csv'
summary.to_csv(summary_csv)
print(f"\nSummary saved to: {summary_csv}")

# Create additional summary: community statistics
print(f"\n{'='*80}")
print("Community Statistics")
print(f"{'='*80}")
community_stats = results_df.groupby(['granularity', 'year', 'community']).agg({
    'node_id': 'count',
    'flow_strength': 'mean',
    'region_name': lambda x: ', '.join(x.head(3))  # Top 3 members
}).rename(columns={'node_id': 'community_size', 'flow_strength': 'avg_flow_strength', 
                   'region_name': 'top_members'})

# Save community stats
community_csv = OUTPUT_DIR / 'community_statistics.csv'
community_stats.to_csv(community_csv)
print(f"Community statistics saved to: {community_csv}")
print(f"Sample:")
print(community_stats.head(20))


Summary by Granularity, Year and Matrix Type:
                                        num_nodes  strength_min  strength_max  \
granularity           year matrix_type                                          
economical_county     1980 flows             2697           0.0           4.0   
                      1985 flows             2697           0.0           7.0   
                      1990 flows             2697           0.0          24.0   
                      1995 flows             2697           0.0          51.0   
                      2000 flows             2697           0.0         140.0   
                      2005 flows             2697           0.0         127.0   
                      2010 flows             2697           0.0         209.0   
                      2015 flows             2697           0.0         325.0   
                      full flows             2697           1.0        3753.0   
economical_prefecture 1980 flows              331           0.

## Documentation

### Overview
This notebook extends the hyperbolic embedding analysis to multiple migration characteristics:
- **flows**: Migration flow counts (TEU = sum of flows)
- **edu_level**: Education level of migrants (TEU = mean education level)
- **family_size**: Family size of migrants (TEU = mean family size)
- **gender**: Gender distribution (TEU = mean gender characteristic)

### Key Innovation: Dual-Source Hyperbolic Coordinates
The hyperbolic embedding explicitly separates the two coordinate sources:

#### Radial Coordinate (r): **FLOWS MATRIX**
- Represents **node strength** = total migration flow through the region
- Computed as: `flow_in + flow_out` for each node
- **Interpretation**: Distance from center = migration centrality
  - Small r (center) = low migration activity
  - Large r (periphery) = high migration activity

#### Angular Coordinate (θ): **DISTANCE MATRIX**
- Represents **Manhattan distance** between regions (precomputed)
- Computed using MATLAB coalescent embedding on distance matrix
- **Interpretation**: Angular position = geographic/economic similarity
  - Nearby angles = similar distance patterns to other regions
  - Distant angles = different distance patterns

### Key Features
1. **Dual-Source Coordinates**: 
   - r from **flows** (migration strength)
   - θ from **distances** (geographic similarity)
2. **Different TEU Calculations**:
   - **flows**: TEU = total flow through node (sum of edge weights)
   - **edu_level, family_size, gender**: TEU = mean characteristic of edges
3. **Top 10 Labeling**: Chinese names + TEU values + radial position
4. **Geographic Mapping**: TEU overlaid on actual China map
5. **Comprehensive Output**: Multiple visualizations per characteristic

### Hyperbolic Coordinates Interpretation
```
Polar coordinates (θ, r):
- θ (angle): From DISTANCE matrix → geographic clustering
- r (radius): From FLOWS matrix → migration centrality

Cartesian coordinates (x, y):
- x = r × cos(θ)
- y = r × sin(θ)
```

**Visual Interpretation:**
- **Center of disk (r ≈ 0)**: Low migration flow regions
- **Edge of disk (r ≈ max)**: High migration flow regions  
- **Angular proximity**: Geographically/economically similar regions
- **Node color**: TEU value for specific characteristic

### Output Files
For each year and matrix type:
- `china_network_{year}_{matrix_type}_hyperbolic.png` - Hyperbolic disk (r from flows, θ from distance, color from TEU)
- `china_map_{year}_{matrix_type}_teu.png` - Geographic map with TEU overlay
- `teu_comparison_{year}.png` - Side-by-side TEU distributions

Summary files:
- `china_network_hyperbolic_by_characteristic.csv` - All data including flow_strength
- `china_network_summary_by_characteristic.csv` - Summary statistics

### Visualization Types

#### 1. Hyperbolic Network (Dual-Source Embedding)
- **Position**: r from flows, θ from Manhattan distance
- **Color**: TEU value (blue → cyan → yellow → red)
- **Labels**: Top 10 nodes with Chinese names, TEU, and r value
- Shows how migration characteristics distribute across flow-distance space

#### 2. Geographic Map
- **Position**: Actual lon/lat coordinates
- **Color/Size**: TEU values
- **Labels**: Top 10 nodes with rank, Chinese name, TEU
- Shows spatial distribution of characteristics

#### 3. TEU Distribution Comparison
- Side-by-side histograms for all 4 types
- Statistics in Chinese (均值, 中位数, 最大值)

### Data Columns
- `year`: Year of observation
- `matrix_type`: flows, edu_level, family_size, or gender
- `node_id`: Node index
- `region_code`: 6-digit region code
- `region_name`: Chinese name
- `theta`: Angular coordinate (from **distance** matrix)
- `r`: Radial coordinate (from **flows** matrix)
- `x`, `y`: Cartesian coordinates
- `teu`: TEU value (method depends on matrix_type)
- `teu_method`: 'sum' or 'mean'
- `flow_strength`: Node strength from flows (same as r, unnormalized)
- `community`: Community from flows network
- `is_core`: Top 10% by degree in flows
- `gamma`: Hyperbolic parameter
- `lon`, `lat`: Geographic coordinates

### Methodology

#### Step 1: Compute Angular Coordinate (θ)
```python
# Load distance matrix (Manhattan distance)
distance_matrix = load_distance_matrix(year)

# Run MATLAB coalescent embedding on distance matrix
coords_temp, gamma = compute_polar_coordinates(distance_matrix)

# Extract angular coordinate
theta = coords_temp[:, 0]
```

#### Step 2: Compute Radial Coordinate (r)
```python
# Load flows matrix
flows_matrix = load_flows_matrix(year)

# Compute node strength (total in + out flow)
flow_strength = flows_matrix.sum(axis=0) + flows_matrix.sum(axis=1)

# Normalize to hyperbolic radius [0, r_max]
r = normalize(flow_strength) * r_max
```

#### Step 3: Compute TEU for Each Characteristic
```python
# For each characteristic (flows, edu_level, family_size, gender):
if characteristic == 'flows':
    TEU = sum_of_edge_weights  # Total flow
else:
    TEU = mean_of_edge_characteristics  # Average characteristic
```

### Interpretation by Characteristic

- **Flows TEU**: Total migration volume through region
  - High TEU = major migration hub
  - Matches radial position (both from flows)

- **Education TEU**: Average education level of migrants
  - High TEU = educated migrants
  - Independent of radial position

- **Family Size TEU**: Average family size of migrants
  - High TEU = larger families
  - Independent of radial position

- **Gender TEU**: Gender distribution
  - Values represent gender balance
  - Independent of radial position

### How to Use
1. Run all cells in order
2. For each year (2010, 2015, 2016):
   - θ computed from **distance matrix** (once)
   - r computed from **flows matrix** (once)
   - TEU computed for each characteristic (4 times)
3. Compare visualizations:
   - Same (θ, r) position across all characteristics
   - Different TEU colors reveal how characteristics vary across flow-distance space

### Key Insights
This dual-source approach reveals:
1. **Migration centrality** (radial position from flows)
2. **Geographic clustering** (angular position from distance)
3. **Characteristic distributions** (color from TEU)

Example: A region might be:
- Peripheral (high r) = high migration flow
- Clustered with others (similar θ) = similar distance pattern
- High education TEU (color) = educated migrants
- Low family size TEU (color) = small families

## Updated Features (All Granularities & Selected Years)

### Enhancements
This notebook has been **enhanced** to process:
- **All granularities**: economical_county, economical_prefecture, economical_province
- **Selected years**: 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015 (5-year intervals)
- **Complete graph**: 'full' - aggregated matrix from entire period (1980-2016)
- **Dual visualization types**:
  1. **Flow Strength Visualization** (existing): Nodes colored by flow strength
  2. **Community Visualization** (NEW): Nodes colored by community with legends

### New Visualizations

#### 1. Flow Strength Hyperbolic Plot (Existing)
- **Filename**: `{granularity}_{year}_{matrix_type}_hyperbolic_flow.png`
- **Features**:
  - Nodes colored by flow strength (blue → cyan → yellow → red)
  - Top 10 nodes labeled with Chinese names and TEU values
  - Legend showing top 10 regions

#### 2. Community Hyperbolic Plot (NEW)
- **Filename**: `{granularity}_{year}_{matrix_type}_hyperbolic_communities.png`
- **Features**:
  - Nodes colored by community (up to 20 distinct colors)
  - Core nodes marked with star markers (⭐)
  - Node size proportional to flow strength
  - **Left legend**: Top 10 regions overall with community ID
  - **Right legend**: Communities with top 3 members each
  - Clear indication of core nodes (marked with *)

#### 3. Geographic Map (Existing)
- **Filename**: `{granularity}_{year}_{matrix_type}_map_flow_strength.png`
- **Features**:
  - Flow strength overlaid on actual China map
  - Top 10 nodes labeled

### Output Files

#### Main CSV
- **File**: `hyperbolic_embeddings_all_granularities_years.csv`
- **Contents**: All hyperbolic coordinates, flow strengths, community assignments, core node flags
- **Columns**:
  - granularity, year, matrix_type
  - node_id, region_code, region_name
  - theta, r, x, y (hyperbolic coordinates)
  - flow_strength, strength_method
  - community, is_core, n_communities
  - gamma, lon, lat

#### Summary CSVs
- **File**: `hyperbolic_summary_by_granularity_year.csv`
  - Summary statistics by granularity, year, and matrix type
- **File**: `community_statistics.csv`
  - Community-level statistics with top members

### Processing Scale
- **Total combinations**: 3 granularities × 9 years (8 intervals + full) × 1 matrix type = **27 processing runs**
- **Visualizations per run**: 3 (flow hyperbolic, community hyperbolic, geographic map)
- **Total visualizations**: **81 plots**

### Key Differences from Original
1. ✅ **All granularities** instead of just economical_county
2. ✅ **5-year intervals** (1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015) + **'full'** aggregated
3. ✅ **Community visualizations** with colored communities and legends
4. ✅ **Core node identification** with star markers
5. ✅ **Community legends** showing top 3 members per community
6. ✅ **Comprehensive CSV** with all data for all granularities/years
7. ✅ **Community statistics** exported separately

### Runtime Expectations
⚠️ **Note**: Processing all granularities and selected years will take time due to:
- MATLAB hyperbolic embedding computation for each year/granularity
- 27 separate embedding computations
- 81 visualization generations

**Estimated time**: 30-60 minutes depending on system performance

### Year Labels in Data
- Numeric years: 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015
- Complete aggregation: 'full' (represents entire 1980-2016 period)